# SmartPark — train the parking-space model

Trains one of SmartPark's two **space segmentation models** (choose in step 2):
* **camera** — security-camera pictures (`space-seg.onnx`, used in the Camera tab);
* **satellite** — satellite / aerial images of lots (`space-sat.onnx`, used when a lot's satellite image is added in Campus · Staff).
  Pictures are scaled so a space is 16 px wide and degraded to satellite sharpness (a space 7–12 real pixels wide, JPEG, blur).

The model is used by SmartPark's hybrid detector (see `docs/hybrid-detection/RESEARCH.md`):
a U-Net with a MobileNetV2 encoder that labels every pixel as *background*, *space interior* or *space boundary*.
The app turns that into individual spaces with a confidence each, and merges them with its geometric detector.

**Always use Runtime → Run all.** If Colab disconnects (free sessions stop after a while or when idle), the machine is wiped: run all again from the top.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. A full run takes roughly 1–3 hours on a free T4,
depending on how much data you include.

**Honesty rules built in:**
* Lots are split into train / validation / test **by lot** — a lot is never in two splits.
* The best epoch is chosen on *validation* lots; the *test* lots are scored once, at the end. Those test numbers
  are the only accuracy numbers to quote, and they only describe lots like the test lots.
* Every dataset passes a licence check before it is used.

Licences of the code used here: PyTorch (BSD-3), segmentation-models-pytorch (MIT), torchvision ImageNet weights (BSD-3),
ONNX / onnxruntime (MIT), OpenCV (Apache-2.0), Shapely / SciPy (BSD-3).

In [ ]:
#@title 1. Get the code and install
REPO = "https://github.com/hazemalosman08-beep/smartpark.git"  #@param {type:"string"}
BRANCH = "hybrid-detection"  #@param {type:"string"}
import os
if os.path.isdir("/content/smartpark/.git"):   # same machine as before: update the code
    %cd /content/smartpark
    !git fetch -q origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD
else:
    %cd /content
    !git clone -q --depth 1 -b {BRANCH} {REPO} smartpark
    %cd /content/smartpark
!pip -q install -r tools/spacemodel/requirements.txt
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the runtime to a GPU")

## 2. Choose the data

| Dataset | Viewpoint | Licence | Notes |
|---|---|---|---|
| **Your SmartPark exports** | any | yours | *Camera tab → Export for training*. The most valuable data. Put the `.json` files in a Google Drive folder. Only use pictures you have the rights to. |
| ACPDS | oblique (~10 m) | MIT | 293 pictures, unseen lots in its test split |
| PKLot | oblique, 3 cameras | CC BY 4.0 (cite Almeida et al. 2015) | thousands of frames of 3 views: capped per view |
| AccessParkCV | top-down aerial | MIT per its card — **checked below before use** | accessible spaces + aisles; ordinary spaces unverified |
| Synthetic | top-down | generated here | warm-up only; never quote metrics from it |

**Which data for which model**
* **camera:** ACPDS + **PKLot** (turn PKLot on: it is the biggest single gain — about 12,000 labelled frames; download it once from the
  official UFPR page into your Drive) + your SmartPark exports from the Camera tab.
* **satellite:** only **top-down** pictures are used (others are set aside automatically): synthetic lots + your SmartPark exports of
  overhead pictures. There is **no public, commercially usable dataset of satellite lots with every space labelled** that we could find.
  AccessParkCV labels mainly *accessible* spaces (ordinary ones are not all labelled, which would teach the model that they are not spaces)
  and its imagery source is not stated, so it is off. Satellite imagery from map services (Google, Esri…) may not be used for training
  under their terms: use pictures you own (e.g. a drone flight of the campus) for your exports.
  Until real top-down lots are in the training data, expect this model to be weak and quote only its test numbers.

In [ ]:
#@title Data switches
MODEL = "camera"               #@param ["camera", "satellite"]
USE_SMARTPARK_EXPORTS = False  #@param {type:"boolean"}
DRIVE_EXPORT_FOLDER = "/content/drive/MyDrive/smartpark-exports"  #@param {type:"string"}
USE_ACPDS = True               #@param {type:"boolean"}
USE_PKLOT = False              #@param {type:"boolean"}
PKLOT_DIR = "/content/drive/MyDrive/PKLot/PKLotSegmented_or_PKLot"  #@param {type:"string"}
PKLOT_PER_VIEW = 300           #@param {type:"integer"}
USE_ACCESSPARKCV = False       #@param {type:"boolean"}
SYNTHETIC_WARMUP = 1000        #@param {type:"integer"}
# Lots that must be in the TEST split (e.g. your own lots you want a fair score on):
FORCE_TEST_LOTS = ""           #@param {type:"string"}
IMAGE_SIZE = 512               #@param [384, 512, 640] {type:"raw"}
EPOCHS = 60                    #@param {type:"integer"}
ENCODER = "mobilenet_v2"       #@param ["mobilenet_v2", "efficientnet-b0", "resnet18"]
# The app downloads the model: mobilenet_v2 (~17 MB) is the size it is built for.

ALLOWED_LICENCES = {"own", "MIT", "CC-BY-4.0", "generated"}
import os, json, glob, shutil, subprocess
DATA = "/content/data/annotations"; os.makedirs(DATA, exist_ok=True)

In [ ]:
#@title 3. Your SmartPark exports (Google Drive)
if USE_SMARTPARK_EXPORTS:
    from google.colab import drive
    drive.mount("/content/drive")
    n = 0
    for f in glob.glob(os.path.join(DRIVE_EXPORT_FOLDER, "*.json")):
        doc = json.load(open(f))
        if doc.get("format") != "smartpark-annotation/v1":
            continue
        doc.setdefault("source_dataset", "smartpark"); doc.setdefault("licence", "own")
        # Hand-checked spaces only? Export keeps every space; spaces marked
        # corrected=True were fixed by you. Low-confidence automatic spaces that
        # you did not check can be dropped here:
        doc["spaces"] = [s for s in doc["spaces"] if s.get("corrected") or (s.get("confidence") or 1) >= 0.5]
        json.dump(doc, open(os.path.join(DATA, "sp-" + os.path.basename(f)), "w")); n += 1
    print(n, "SmartPark pictures")

In [ ]:
#@title 4. ACPDS (MIT)
if USE_ACPDS:
    !mkdir -p /content/raw/acpds && cd /content/raw/acpds && wget -q -O acpds.zip "https://pub-e8bbdcbe8f6243b2a9933704a9b1d8bc.r2.dev/parking%2Frois_gopro.zip" && unzip -q -o acpds.zip
    root = os.path.dirname(glob.glob("/content/raw/acpds/**/annotations.json", recursive=True)[0])
    !python -m tools.spacemodel.convert acpds {root} {DATA}

In [ ]:
#@title 5. PKLot (CC BY 4.0) — download it yourself from the official UFPR page into Drive, then run
if USE_PKLOT:
    !python -m tools.spacemodel.convert pklot {PKLOT_DIR} {DATA} --per-view {PKLOT_PER_VIEW}

In [ ]:
#@title 6. AccessParkCV — licence and format are checked before anything is used
if USE_ACCESSPARKCV:
    !pip -q install huggingface_hub
    from huggingface_hub import snapshot_download
    path = snapshot_download("makeabilitylab/disabilityparking", repo_type="dataset", local_dir="/content/raw/accessparkcv")
    card = open(os.path.join(path, "README.md")).read()
    print(card[:3000])
    licence_ok = any(k in card.lower() for k in ["license: mit", "licence: mit", "mit license"])
    assert licence_ok, "Licence on the dataset card is not MIT: read it and decide before using this dataset."
    print("\nFiles:", sorted({os.path.splitext(f)[1] for f in glob.glob(path + '/**/*', recursive=True)}))
    # If the labels are YOLO-OBB text files, convert the parking-space classes
    # (look up their ids in the card above and list them here):
    SPACE_CLASS_IDS = ""  #@param {type:"string"}
    if SPACE_CLASS_IDS:
        !python -m tools.spacemodel.convert yolo-obb {path} {DATA} --classes {SPACE_CLASS_IDS} --licence MIT --name accessparkcv

In [ ]:
#@title 7. Licence gate and split by lot (no lot in two splits)
bad = []
for f in glob.glob(os.path.join(DATA, "*.json")):
    if f.endswith("splits.json"): continue
    lic = json.load(open(f)).get("licence", "check")
    if lic not in ALLOWED_LICENCES: bad.append((os.path.basename(f), lic))
assert not bad, f"Remove or clear the licence of: {bad[:5]}"
if MODEL == "satellite":
    # Only top-down pictures teach the satellite model; the rest are set aside.
    os.makedirs("/content/data/set_aside", exist_ok=True)
    moved = 0
    for f in glob.glob(os.path.join(DATA, "*.json")):
        if f.endswith("splits.json"): continue
        if json.load(open(f)).get("view") != "overhead":
            shutil.move(f, "/content/data/set_aside/"); moved += 1
    left = len([f for f in glob.glob(os.path.join(DATA, "*.json")) if not f.endswith("splits.json")])
    print(f"satellite model: {left} top-down pictures kept, {moved} camera-angle pictures set aside")
    assert left >= 10, "Too few top-down pictures: add SmartPark exports of overhead pictures (your own drone / aerial images)."
!python -m tools.spacemodel.split {DATA} --val 0.15 --test 0.2 --test-lots "{FORCE_TEST_LOTS}"

In [ ]:
#@title 8. Look at a few training targets (white = boundary, grey = space)
import sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, ".")
from tools.spacemodel.annotation import load
from tools.spacemodel.masks import sample
files = json.load(open(os.path.join(DATA, "splits.json")))["files"]["train"][:4]
fig, ax = plt.subplots(len(files), 2, figsize=(10, 5 * len(files)))
for i, f in enumerate(files):
    img, mask = sample(load(os.path.join(DATA, f)), IMAGE_SIZE, DATA)
    ax[i][0].imshow(img); ax[i][1].imshow(mask, cmap="gray", vmin=0, vmax=2); ax[i][0].set_title(f)
plt.show()

In [ ]:
#@title 9. (Optional) warm-up on synthetic lots — weights only, its metrics mean nothing
if SYNTHETIC_WARMUP:
    !python -m tools.spacemodel.synth /content/data/synth --count {SYNTHETIC_WARMUP} --lots 200
    !python -m tools.spacemodel.split /content/data/synth
    !python -m tools.spacemodel.train --data /content/data/synth --out /content/runs/synth --size {IMAGE_SIZE} --epochs 8 --batch 16 --domain {MODEL} --encoder {ENCODER}

In [ ]:
#@title 10. Train on the real data
init = "--init /content/runs/synth/best.pt" if SYNTHETIC_WARMUP and os.path.exists("/content/runs/synth/best.pt") else ""
!python -m tools.spacemodel.train --data {DATA} --out /content/runs/space --size {IMAGE_SIZE} --epochs {EPOCHS} --batch 8 --domain {MODEL} --encoder {ENCODER} {init}

In [ ]:
#@title 11. Test-lot results (lots never used for training or for choosing the epoch)
m = json.load(open("/content/runs/space/metrics.json"))
print("Data:", m["data"]["train"], "train /", m["data"]["val"], "val /", m["data"]["test"], "test pictures")
print("Best validation F1:", round(m["best_val_f1"], 3))
if m["test"]:
    print("TEST overall:", m["test"]["overall"])
    for lot, r in m["test"]["per_lot"].items():
        print(f"  {lot:30} P {r['precision']:.3f}  R {r['recall']:.3f}  F1 {r['f1']:.3f}")
else:
    print("No test lots - add more lots before trusting this model.")

In [ ]:
#@title 11b. Re-score the same model with each way of cutting its picture into spaces (no retraining)
!cd /content/smartpark && git fetch -q origin {BRANCH} && git reset -q --hard FETCH_HEAD
!cd /content/smartpark && python -m tools.spacemodel.evaluate_run --data {DATA} --weights /content/runs/space/best.pt --size {IMAGE_SIZE} --domain {MODEL} --encoder {ENCODER}

In [ ]:
#@title 12. Export for the app and download
!python -m tools.spacemodel.export --weights /content/runs/space/best.pt --metrics /content/runs/space/metrics.json --out /content/export --size {IMAGE_SIZE} --domain {MODEL} --encoder {ENCODER} --datasets "{'smartpark' if USE_SMARTPARK_EXPORTS else ''},{'acpds(MIT)' if USE_ACPDS else ''},{'pklot(CC-BY-4.0)' if USE_PKLOT else ''},{'accessparkcv(MIT)' if USE_ACCESSPARKCV else ''}"
from google.colab import files
NAME = "space-sat" if MODEL == "satellite" else "space-seg"
files.download(f"/content/export/{NAME}.onnx"); files.download(f"/content/export/{NAME}.json")

## 13. Put it in the app — and check it before deploying

1. Copy the two downloaded files (`space-seg.*` for the camera model, `space-sat.*` for the satellite model) into
   `public/models/` of the `hybrid-detection` branch.
2. Run the regression suite: `npm run test:accuracy`. It must report **no regressions** against
   `test/accuracy/baseline.json`; ideally it now finds more spaces (e.g. on `yellow.png`).
3. Only then merge to `main` (that is what Netlify deploys).

Without these two files the app behaves exactly as before (geometric detection only).